# TP4 — De Cassandra à Spark : analyse de la saison F1 2024

**M2 Big Data & IA — Données distribuées**

Données métier : projet Cassandra des TP2/TP3, alimenté par l'API [OpenF1](https://openf1.org).

- Keyspace : **`f1`** (`NetworkTopologyStrategy`, `dc1 = 3`, cluster `cass1` / `cass2` / `cass3`)
- Table : **`results_by_driver`** — résultat de chaque pilote à chaque Grand Prix 2024

**Question métier : quels pilotes et quelles écuries ont dominé la saison 2024, et avec quelle régularité (podiums, abandons) ?**

### Préparation (terminal hôte)
```bash
docker exec -it cass1 nodetool status              # 3 nœuds UN
docker network connect cass-net spark-tp1          # Spark rejoint le réseau du cluster
docker network inspect cass-net --format '{{range .Containers}}{{.Name}} -> {{.IPv4Address}}{{"\n"}}{{end}}'
```

## Partie 1 — Connecter Spark à Cassandra
### SparkSession

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TP4-Cassandra-Spark-F1")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

sc = spark.sparkContext

print("Version Spark :", spark.version)
print("Master :", sc.master)
print("Parallélisme :", sc.defaultParallelism)

Version Spark : 4.2.0
Master : local[4]
Parallélisme : 4


`local[4]` : Spark s'exécute dans le conteneur avec 4 threads ; `shuffle.partitions = 4` remplace la valeur par défaut (200), trop élevée pour ~500 lignes.

### Connexion à Cassandra

In [2]:
%pip install -q cassandra-driver

Note: you may need to restart the kernel to use updated packages.


In [3]:
from cassandra.cluster import Cluster

cluster = Cluster(["cass1"])
session = cluster.connect()
print("Connexion Cassandra réussie")

Connexion Cassandra réussie


In [4]:
rows = session.execute("""
    SELECT keyspace_name
    FROM system_schema.keyspaces
""")

for row in rows:
    print(row.keyspace_name)

system_auth
system_schema
f1
system_distributed
system
system_traces


### Keyspace, table et structure

In [5]:
KEYSPACE = "f1"
TABLE = "results_by_driver"

session.set_keyspace(KEYSPACE)
print("Keyspace :", KEYSPACE)
print("Table    :", TABLE)

rows = session.execute("""
    SELECT column_name, type, kind
    FROM system_schema.columns
    WHERE keyspace_name = %s
      AND table_name = %s
""", (KEYSPACE, TABLE))

for row in rows:
    print(f"  {row.column_name} -> {row.type} ({row.kind})")

Keyspace : f1
Table    : results_by_driver
  date_start -> timestamp (clustering)
  driver_number -> int (partition_key)
  full_name -> text (regular)
  location -> text (regular)
  points -> double (regular)
  position -> int (regular)
  session_key -> int (clustering)
  status -> text (regular)
  team_name -> text (regular)
  year -> int (partition_key)


Clé primaire : `((driver_number, year), date_start, session_key)` — une partition Cassandra par pilote et par saison, courses triées par date (requête du TP2 : « résultats d'un pilote sur une saison »).

### Quelques lignes

In [6]:
rows = session.execute(f"SELECT * FROM {TABLE} LIMIT 5")

for row in rows:
    print(row)

Row(driver_number=61, year=2024, date_start=datetime.datetime(2024, 12, 8, 13, 0), session_key=9662, full_name='Jack DOOHAN', location='Yas Island', points=0.0, position=15, status='FINISHED', team_name='Alpine')
Row(driver_number=38, year=2024, date_start=datetime.datetime(2024, 3, 9, 17, 0), session_key=9480, full_name='Oliver BEARMAN', location='Jeddah', points=6.0, position=7, status='FINISHED', team_name='Ferrari')
Row(driver_number=3, year=2024, date_start=datetime.datetime(2024, 3, 2, 15, 0), session_key=9472, full_name='Daniel RICCIARDO', location='Sakhir', points=0.0, position=13, status='FINISHED', team_name='RB')
Row(driver_number=3, year=2024, date_start=datetime.datetime(2024, 3, 9, 17, 0), session_key=9480, full_name='Daniel RICCIARDO', location='Jeddah', points=0.0, position=16, status='FINISHED', team_name='RB')
Row(driver_number=3, year=2024, date_start=datetime.datetime(2024, 3, 24, 4, 0), session_key=9488, full_name='Daniel RICCIARDO', location='Melbourne', points=0.

## Partie 2 — Créer le DataFrame Spark
Cassandra → données récupérées (driver Python) → Spark DataFrame

In [7]:
rows = session.execute(f"SELECT * FROM {TABLE}")

columns = rows.column_names
data = [tuple(row) for row in rows]

print("Colonnes :", columns)
print("Nombre de lignes récupérées :", len(data))

Colonnes : ['driver_number', 'year', 'date_start', 'session_key', 'full_name', 'location', 'points', 'position', 'status', 'team_name']
Nombre de lignes récupérées : 479


In [8]:
df = spark.createDataFrame(data, columns)

df.show(10, truncate=False)
df.printSchema()
print("Nombre de lignes dans le DataFrame :", df.count())

+-------------+----+-------------------+-----------+----------------+----------+------+--------+--------+---------+
|driver_number|year|date_start         |session_key|full_name       |location  |points|position|status  |team_name|
+-------------+----+-------------------+-----------+----------------+----------+------+--------+--------+---------+
|61           |2024|2024-12-08 13:00:00|9662       |Jack DOOHAN     |Yas Island|0.0   |15      |FINISHED|Alpine   |
|38           |2024|2024-03-09 17:00:00|9480       |Oliver BEARMAN  |Jeddah    |6.0   |7       |FINISHED|Ferrari  |
|3            |2024|2024-03-02 15:00:00|9472       |Daniel RICCIARDO|Sakhir    |0.0   |13      |FINISHED|RB       |
|3            |2024|2024-03-09 17:00:00|9480       |Daniel RICCIARDO|Jeddah    |0.0   |16      |FINISHED|RB       |
|3            |2024|2024-03-24 04:00:00|9488       |Daniel RICCIARDO|Melbourne |0.0   |12      |FINISHED|RB       |
|3            |2024|2024-04-07 05:00:00|9496       |Daniel RICCIARDO|Suz

Nombre de lignes dans le DataFrame : 479


**Données utilisées** : les 479 résultats de course de la saison 2024 (24 Grands Prix × ~20 pilotes, remplaçants inclus).

**Colonnes importantes** :
- `full_name`, `team_name` : le pilote et son écurie (axes d'agrégation) ;
- `position`, `status` : place à l'arrivée et statut (`FINISHED`, `DNF`, `DNS`, `DSQ`) ;
- `points` : points marqués sur la course (Grand Prix uniquement, hors sprints) ;
- `location`, `date_start` : le Grand Prix concerné.

**Information métier analysée** : la domination (points) et la régularité (podiums, abandons) des pilotes et écuries sur la saison.

## Partie 3 — Transformations
### Sélection

In [9]:
from pyspark.sql import functions as F

df_sel = df.select("full_name", "team_name", "location", "date_start", "position", "status", "points")
df_sel.show(5, truncate=False)

+----------------+---------+----------+-------------------+--------+--------+------+
|full_name       |team_name|location  |date_start         |position|status  |points|
+----------------+---------+----------+-------------------+--------+--------+------+
|Jack DOOHAN     |Alpine   |Yas Island|2024-12-08 13:00:00|15      |FINISHED|0.0   |
|Oliver BEARMAN  |Ferrari  |Jeddah    |2024-03-09 17:00:00|7       |FINISHED|6.0   |
|Daniel RICCIARDO|RB       |Sakhir    |2024-03-02 15:00:00|13      |FINISHED|0.0   |
|Daniel RICCIARDO|RB       |Jeddah    |2024-03-09 17:00:00|16      |FINISHED|0.0   |
|Daniel RICCIARDO|RB       |Melbourne |2024-03-24 04:00:00|12      |FINISHED|0.0   |
+----------------+---------+----------+-------------------+--------+--------+------+
only showing top 5 rows


### Filtres métier

In [10]:
df.groupBy("status").count().show()

+--------+-----+
|  status|count|
+--------+-----+
|FINISHED|  425|
|     DNF|   49|
|     DNS|    3|
|     DSQ|    2|
+--------+-----+



Les pilotes non classés ont une position technique ≥ 100 issue de l'import : on les traite à part avec `status`.

In [11]:
# Filtre 1 : les podiums (pilotes classés dans les 3 premiers)
podiums = df_sel.filter((F.col("status") == "FINISHED") & (F.col("position") <= 3))
podiums.orderBy("date_start", "position").show(9, truncate=False)

+---------------+---------------+---------+-------------------+--------+--------+------+
|full_name      |team_name      |location |date_start         |position|status  |points|
+---------------+---------------+---------+-------------------+--------+--------+------+
|Max VERSTAPPEN |Red Bull Racing|Sakhir   |2024-03-02 15:00:00|1       |FINISHED|26.0  |
|Sergio PEREZ   |Red Bull Racing|Sakhir   |2024-03-02 15:00:00|2       |FINISHED|18.0  |
|Carlos SAINZ   |Ferrari        |Sakhir   |2024-03-02 15:00:00|3       |FINISHED|15.0  |
|Max VERSTAPPEN |Red Bull Racing|Jeddah   |2024-03-09 17:00:00|1       |FINISHED|25.0  |
|Sergio PEREZ   |Red Bull Racing|Jeddah   |2024-03-09 17:00:00|2       |FINISHED|18.0  |
|Charles LECLERC|Ferrari        |Jeddah   |2024-03-09 17:00:00|3       |FINISHED|16.0  |
|Carlos SAINZ   |Ferrari        |Melbourne|2024-03-24 04:00:00|1       |FINISHED|25.0  |
|Charles LECLERC|Ferrari        |Melbourne|2024-03-24 04:00:00|2       |FINISHED|19.0  |
|Lando NORRIS   |McLa

In [12]:
# Filtre 2 : les abandons / non-partants / disqualifications
non_classes = df_sel.filter(F.col("status") != "FINISHED")
non_classes.show(10, truncate=False)

+----------------+-----------+-----------------+-------------------+--------+------+------+
|full_name       |team_name  |location         |date_start         |position|status|points|
+----------------+-----------+-----------------+-------------------+--------+------+------+
|Daniel RICCIARDO|RB         |Suzuka           |2024-04-07 05:00:00|118     |DNF   |0.0   |
|Daniel RICCIARDO|RB         |Shanghai         |2024-04-21 07:00:00|117     |DNF   |0.0   |
|Charles LECLERC |Ferrari    |Montréal         |2024-06-09 18:00:00|118     |DNF   |0.0   |
|Valtteri BOTTAS |Kick Sauber|Shanghai         |2024-04-21 07:00:00|119     |DNF   |0.0   |
|Valtteri BOTTAS |Kick Sauber|Yas Island       |2024-12-08 13:00:00|117     |DNF   |0.0   |
|ZHOU Guanyu     |Kick Sauber|Suzuka           |2024-04-07 05:00:00|117     |DNF   |0.0   |
|ZHOU Guanyu     |Kick Sauber|Spa-Francorchamps|2024-07-28 13:00:00|118     |DNF   |0.0   |
|Pierre GASLY    |Alpine     |Jeddah           |2024-03-09 17:00:00|119     |DNF

### Colonnes calculées
- `pct_points_victoire` : part des points d'une victoire (25 pts) obtenue sur la course — > 100 % quand le vainqueur prend aussi le point du meilleur tour ;
- `categorie` : niveau de résultat (Podium / Points / Hors points / Non classé).

In [13]:
POINTS_VICTOIRE = 25

df_calc = (
    df_sel
    .withColumn("pct_points_victoire", F.round(F.col("points") / POINTS_VICTOIRE * 100, 1))
    .withColumn(
        "categorie",
        F.when(F.col("status") != "FINISHED", "Non classé")
         .when(F.col("position") <= 3, "Podium")
         .when(F.col("position") <= 10, "Points")
         .otherwise("Hors points")
    )
)

df_calc.orderBy(F.desc("points")).show(10, truncate=False)

+--------------+---------------+-----------------+-------------------+--------+--------+------+-------------------+---------+
|full_name     |team_name      |location         |date_start         |position|status  |points|pct_points_victoire|categorie|
+--------------+---------------+-----------------+-------------------+--------+--------+------+-------------------+---------+
|Max VERSTAPPEN|Red Bull Racing|Sakhir           |2024-03-02 15:00:00|1       |FINISHED|26.0  |104.0              |Podium   |
|Max VERSTAPPEN|Red Bull Racing|Suzuka           |2024-04-07 05:00:00|1       |FINISHED|26.0  |104.0              |Podium   |
|Max VERSTAPPEN|Red Bull Racing|São Paulo        |2024-11-03 15:30:00|1       |FINISHED|26.0  |104.0              |Podium   |
|Lando NORRIS  |McLaren        |Zandvoort        |2024-08-25 13:00:00|1       |FINISHED|26.0  |104.0              |Podium   |
|Lewis HAMILTON|Mercedes       |Silverstone      |2024-07-07 14:00:00|1       |FINISHED|25.0  |100.0              |Pod

## Partie 4 — Actions et agrégations

In [14]:
# Action de comptage
print("Résultats :", df_calc.count())
print("Podiums   :", podiums.count())
print("Non classés :", non_classes.count())

Résultats : 479


Podiums   : 72


Non classés : 54


In [15]:
# Métriques globales sur les pilotes classés
df_calc.filter(F.col("status") == "FINISHED").select(
    F.min("points").alias("minimum_points"),
    F.max("points").alias("maximum_points"),
    F.round(F.avg("points"), 2).alias("moyenne_points"),
    F.round(F.avg("position"), 2).alias("position_moyenne")
).show()

+--------------+--------------+--------------+----------------+
|minimum_points|maximum_points|moyenne_points|position_moyenne|
+--------------+--------------+--------------+----------------+
|           0.0|          26.0|          5.75|            9.43|
+--------------+--------------+--------------+----------------+



In [16]:
# Combien de résultats dans chaque catégorie ?
df_calc.groupBy("categorie").count().orderBy(F.desc("count")).show()

+-----------+-----+
|  categorie|count|
+-----------+-----+
|Hors points|  185|
|     Points|  168|
|     Podium|   72|
| Non classé|   54|
+-----------+-----+



In [17]:
# Quelle écurie a marqué le plus de points ? (classement constructeurs, hors sprints)
df_calc.groupBy("team_name") \
    .agg(
        F.sum("points").alias("points"),
        F.count("*").alias("departs"),
        F.sum(F.when(F.col("categorie") == "Podium", 1).otherwise(0)).alias("podiums"),
    ) \
    .orderBy(F.desc("points")) \
    .show(truncate=False)

+---------------+------+-------+-------+
|team_name      |points|departs|podiums|
+---------------+------+-------+-------+
|McLaren        |609.0 |48     |21     |
|Ferrari        |595.0 |48     |22     |
|Red Bull Racing|537.0 |48     |18     |
|Mercedes       |433.0 |48     |9      |
|Aston Martin   |94.0  |48     |0      |
|Alpine         |63.0  |48     |2      |
|Haas F1 Team   |51.0  |48     |0      |
|RB             |40.0  |48     |0      |
|Williams       |17.0  |47     |0      |
|Kick Sauber    |4.0   |48     |0      |
+---------------+------+-------+-------+



## Partie 5 — Partitions et Lazy Evaluation
### Partitions

In [18]:
print("Nombre de partitions initiales :", df.rdd.getNumPartitions())
print("Lignes par partition :", df.rdd.glom().map(len).collect())

df8 = df.repartition(8)
print("Nombre de partitions après repartition(8) :", df8.rdd.getNumPartitions())
print("Lignes par partition :", df8.rdd.glom().map(len).collect())

df2 = df8.coalesce(2)
print("Nombre de partitions après coalesce(2) :", df2.rdd.getNumPartitions())
print("Lignes par partition :", df2.rdd.glom().map(len).collect())

Nombre de partitions initiales : 4


Lignes par partition : [119, 119, 119, 122]
Nombre de partitions après repartition(8) : 8


Lignes par partition : [60, 58, 60, 60, 60, 60, 61, 60]


Nombre de partitions après coalesce(2) : 2
Lignes par partition : [241, 238]


- `createDataFrame` découpe la liste Python en `defaultParallelism` = 4 partitions (une tâche par partition).
- `repartition(n)` redistribue toutes les lignes via un **shuffle** (répartition équilibrée).
- `coalesce(n)` réduit le nombre de partitions **sans shuffle** en fusionnant des partitions voisines.

### Lazy Evaluation : classement du championnat pilotes

In [19]:
import time

t0 = time.time()
etape1 = df.filter(F.col("points") > 0)                                       # transformation
etape2 = etape1.groupBy("full_name", "team_name")                             # transformation
etape3 = etape2.agg(F.sum("points").alias("total_points"))                    # transformation
championnat = etape3.orderBy(F.desc("total_points"))                          # transformation

print(f"4 transformations déclarées en {time.time() - t0:.3f} s : aucun job Spark lancé.")
print("Jobs déjà exécutés :", len(sc.statusTracker().getJobIdsForGroup()))

4 transformations déclarées en 0.011 s : aucun job Spark lancé.
Jobs déjà exécutés : 27


In [20]:
# Action : déclenche l'exécution du DAG
championnat.show(10, truncate=False)
print("Jobs exécutés après l'action :", len(sc.statusTracker().getJobIdsForGroup()))

+---------------+---------------+------------+
|full_name      |team_name      |total_points|
+---------------+---------------+------------+
|Max VERSTAPPEN |Red Bull Racing|399.0       |
|Lando NORRIS   |McLaren        |344.0       |
|Charles LECLERC|Ferrari        |327.0       |
|Oscar PIASTRI  |McLaren        |265.0       |
|Carlos SAINZ   |Ferrari        |262.0       |
|George RUSSELL |Mercedes       |226.0       |
|Lewis HAMILTON |Mercedes       |207.0       |
|Sergio PEREZ   |Red Bull Racing|138.0       |
|Fernando ALONSO|Aston Martin   |70.0        |
|Pierre GASLY   |Alpine         |40.0        |
+---------------+---------------+------------+
only showing top 10 rows
Jobs exécutés après l'action : 29


**Pourquoi Spark utilise-t-il la Lazy Evaluation ?**

Les transformations ne font qu'enrichir un plan logique. Quand une action est appelée, Spark connaît la chaîne complète et l'optimiseur **Catalyst** peut :
- réordonner et pousser les filtres au plus près de la source (*predicate pushdown*) ;
- ne lire que les colonnes utiles (*column pruning* : le plan ci-dessous ne garde que `full_name`, `team_name`, `points`) ;
- fusionner les opérations d'un même stage en un seul passage sur les données (*whole-stage codegen*) ;
- éviter de calculer des résultats intermédiaires qui ne seront jamais utilisés.

Résultat : moins de lectures, moins de shuffle et un seul DAG exécuté par action.

In [21]:
championnat.explain(True)

== Parsed Logical Plan ==
'Sort ['total_points DESC NULLS LAST], true
+- Aggregate [full_name#4, team_name#9], [full_name#4, team_name#9, sum(points#6) AS total_points#302]
   +- Filter (points#6 > cast(0 as double))
      +- LogicalRDD [driver_number#0L, year#1L, date_start#2, session_key#3L, full_name#4, location#5, points#6, position#7L, status#8, team_name#9], false

== Analyzed Logical Plan ==
full_name: string, team_name: string, total_points: double
Sort [total_points#302 DESC NULLS LAST], true
+- Aggregate [full_name#4, team_name#9], [full_name#4, team_name#9, sum(points#6) AS total_points#302]
   +- Filter (points#6 > cast(0 as double))
      +- LogicalRDD [driver_number#0L, year#1L, date_start#2, session_key#3L, full_name#4, location#5, points#6, position#7L, status#8, team_name#9], false

== Optimized Logical Plan ==
Sort [total_points#302 DESC NULLS LAST], true
+- Aggregate [full_name#4, team_name#9], [full_name#4, team_name#9, sum(points#6) AS total_points#302]
   +- Proje

## Partie 6 — Spark UI et Shuffle
Spark UI : **http://localhost:4040** (ou 4041 si une autre session Spark occupe déjà le port).

Plan physique d'une agrégation : la ligne `Exchange hashpartitioning` matérialise le **shuffle**.

In [22]:
df_calc.groupBy("categorie").count().explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[categorie#143], functions=[count(1)])
   +- Exchange hashpartitioning(categorie#143, 4), ENSURE_REQUIREMENTS, [plan_id=549]
      +- HashAggregate(keys=[categorie#143], functions=[partial_count(1)])
         +- Project [CASE WHEN NOT (status#8 = FINISHED) THEN Non classé WHEN (position#7L <= 3) THEN Podium WHEN (position#7L <= 10) THEN Points ELSE Hors points END AS categorie#143]
            +- Scan ExistingRDD[driver_number#0L,year#1L,date_start#2,session_key#3L,full_name#4,location#5,points#6,position#7L,status#8,team_name#9]




In [23]:
# Relance de l'action pour l'observer dans l'onglet Jobs / Stages
sc.setJobDescription("Classement championnat pilotes 2024")
championnat.show(10, truncate=False)
sc.setJobDescription(None)

+---------------+---------------+------------+
|full_name      |team_name      |total_points|
+---------------+---------------+------------+
|Max VERSTAPPEN |Red Bull Racing|399.0       |
|Lando NORRIS   |McLaren        |344.0       |
|Charles LECLERC|Ferrari        |327.0       |
|Oscar PIASTRI  |McLaren        |265.0       |
|Carlos SAINZ   |Ferrari        |262.0       |
|George RUSSELL |Mercedes       |226.0       |
|Lewis HAMILTON |Mercedes       |207.0       |
|Sergio PEREZ   |Red Bull Racing|138.0       |
|Fernando ALONSO|Aston Martin   |70.0        |
|Pierre GASLY   |Alpine         |40.0        |
+---------------+---------------+------------+
only showing top 10 rows


Observations dans la Spark UI pour `championnat.show(10)` (captures dans `screenshots/`) :

| Élément | Observation |
|---|---|
| **Jobs** | l'action `show()` déclenche **2 jobs** (29 et 30, description « Classement championnat pilotes 2024 ») ; la session complète en compte 33 |
| **Stage 40** (job 29) | lecture du DataFrame + `filter` + `project` + agrégation partielle : **4 tasks** (une par partition), 63 ms, **Shuffle Write 1 784 B / 24 enregistrements** |
| **Stage 41** (job 30) | *skipped* : le résultat du shuffle est réutilisé |
| **Stage 42** (job 30) | agrégation finale + tri : **1 task**, 5 ms, **Shuffle Read 1 784 B / 24 enregistrements** |
| **Executors** | un seul executor (`driver`) avec 4 cœurs : mode `local[4]` |

- L'`Exchange hashpartitioning` du `groupBy` coupe le DAG en deux stages : c'est là qu'apparaissent Shuffle Write puis Shuffle Read.
- L'agrégation partielle (`partial_sum`) réduit 240 lignes filtrées à 24 lignes avant le shuffle : seules 24 lignes traversent le réseau.
- Avec l'**AQE** (Adaptive Query Execution), les 4 partitions de shuffle presque vides sont fusionnées en 1 seule (`AQEShuffleRead`) : 1 task au lieu de 4.
- `orderBy` + `show(10)` est remplacé à l'exécution par un `TakeOrderedAndProject` (top 10) : le second `Exchange rangepartitioning` visible dans `explain()` n'est pas exécuté.

In [24]:
bilan_pilotes = (
    df.select("full_name", "team_name", "position", "status", "points")       # sélection
      .filter(F.col("status") != "DNS")                                          # filtre : courses réellement prises
      .withColumn("classe", (F.col("status") == "FINISHED").cast("int"))        # colonne calculée
      .withColumn("podium", ((F.col("status") == "FINISHED") & (F.col("position") <= 3)).cast("int"))
      .groupBy("full_name", "team_name")                                         # agrégation
      .agg(
          F.count("*").alias("courses"),
          F.sum("points").alias("points"),
          F.sum("podium").alias("podiums"),
          F.sum(1 - F.col("classe")).alias("non_classe"),
          F.round(F.avg(F.when(F.col("classe") == 1, F.col("position"))), 1).alias("position_moy"),
      )
      .withColumn("points_par_course", F.round(F.col("points") / F.col("courses"), 2))
      .withColumn("taux_podium_pct", F.round(F.col("podiums") / F.col("courses") * 100, 1))
      .orderBy(F.desc("points"))
)

bilan_pilotes.show(12, truncate=False)                                           # action

+---------------+---------------+-------+------+-------+----------+------------+-----------------+---------------+
|full_name      |team_name      |courses|points|podiums|non_classe|position_moy|points_par_course|taux_podium_pct|
+---------------+---------------+-------+------+-------+----------+------------+-----------------+---------------+
|Max VERSTAPPEN |Red Bull Racing|24     |399.0 |14     |1         |3.0         |16.63            |58.3           |
|Lando NORRIS   |McLaren        |24     |344.0 |13     |0         |4.3         |14.33            |54.2           |
|Charles LECLERC|Ferrari        |24     |327.0 |13     |1         |3.9         |13.63            |54.2           |
|Oscar PIASTRI  |McLaren        |24     |265.0 |8      |0         |5.1         |11.04            |33.3           |
|Carlos SAINZ   |Ferrari        |23     |262.0 |9      |3         |4.1         |11.39            |39.1           |
|George RUSSELL |Mercedes       |24     |226.0 |4      |3         |5.0         |

**Lecture des résultats**
- **Max Verstappen** domine : le plus de points (399), le plus de podiums (14 sur 24, soit 58 %) et la meilleure position moyenne (3,0).
- **Lando Norris** et **Charles Leclerc** suivent avec 13 podiums chacun ; Norris est le seul du top 3 à avoir vu l'arrivée de toutes les courses.
- **McLaren** gagne le classement des écuries (609 pts) grâce à deux pilotes réguliers (Norris + Piastri), alors que **Red Bull** (537 pts) repose surtout sur Verstappen : Pérez ne marque que 138 pts avec 5 courses non terminées.
- **Ferrari** est 2ᵉ en points (595) mais compte le plus de podiums (22).

Les totaux diffèrent du classement officiel car la table ne contient que les Grands Prix (pas les courses sprint).

In [25]:
cluster.shutdown()